In [1]:
# ============================================================
# 1. INSTALL  (no -U! upgrading numpy mid-session breaks the kernel)
# ============================================================
!pip -q install requests tqdm pymupdf rank-bm25 sentence-transformers chromadb transformers accelerate sentencepiece scikit-learn google-genai

print("✅ Installed.")
print("⚠️ NOW RESTART THE SESSION (Kaggle: Run → Restart & clear outputs | Colab: Runtime → Restart)")
print("   Then continue from cell 2. You only do this ONCE.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 79.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 120.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 21.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/6

In [2]:
# ============================================================
# 2. CONFIG
# ============================================================
import os, re, json, time, requests
import numpy as np
import pandas as pd
import torch
import pymupdf as fitz   # fitz API deprecated — same functions, new name
from pathlib import Path
from tqdm.auto import tqdm
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# --- Embedding model: auto-select by hardware ---
if torch.cuda.is_available():
    EMBEDDING_MODEL = "BAAI/bge-m3"             # strong, MIT license, needs GPU
    print("🖥️  GPU detected → embedding with BGE-M3")
else:
    EMBEDDING_MODEL = "BAAI/bge-base-en-v1.5"   # strong English model, CPU-friendly
    print("🐢 No GPU → embedding with bge-base-en-v1.5 (enable GPU for BGE-M3)")

RERANKER_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
GENERATOR_MODEL = "google/flan-t5-base"          # local fallback
GEMINI_MODEL    = "gemini-3.5-flash-lite"

N_PAPERS_TARGET = 80
MAX_PAPERS_PER_QUERY = 20
MIN_ABSTRACT_CHARS = 120

SEARCH_QUERIES = [
    '"machine learning"', '"deep learning"', '"artificial intelligence"',
    '"large language models"', '"transformers"',
    '"retrieval augmented generation"', '"computer vision"',
    '"natural language processing"',
]

# --- Chunking (tuned for academic papers) ---
CHUNK_SIZE_WORDS   = 220
CHUNK_OVERLAP_WORDS = 40
MIN_CHUNK_WORDS    = 40

# --- Retrieval funnel: cheap retrieval → expensive rerank ---
BM25_TOP_K   = 30
DENSE_TOP_K  = 30
HYBRID_TOP_K = 40
RERANK_TOP_K = 10

# --- Paths (Kaggle / Colab / local) ---
if os.path.exists("/kaggle/working"):
    BASE_DIR = Path("/kaggle/working/academic_rag")
else:
    BASE_DIR = Path("/content/academic_rag")
PDF_DIR      = BASE_DIR / "pdfs"
ARTIFACT_DIR = BASE_DIR / "artifacts"
CHROMA_DIR   = BASE_DIR / "chroma"
for p in [PDF_DIR, ARTIFACT_DIR, CHROMA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

PAPERS_JSON    = ARTIFACT_DIR / "papers.json"
CHUNKS_JSON    = ARTIFACT_DIR / "chunks.json"
EMBEDDINGS_NPY = ARTIFACT_DIR / "embeddings.npy"
RUN_CFG_JSON   = ARTIFACT_DIR / "run_config.json"

S2_API_KEY = os.getenv("SEMANTIC_SCHOLAR_API_KEY", "")
HEADERS = {"User-Agent": "AcademicRAGDemo/1.0"}
if S2_API_KEY:
    HEADERS["x-api-key"] = S2_API_KEY

print("✅ Config loaded |", BASE_DIR)

🖥️  GPU detected → embedding with BGE-M3
✅ Config loaded | /content/academic_rag


In [3]:
# ============================================================
# 3. SEARCH ARXIV  (no API key, no rate-limit pain)
#    Same output schema as before — downstream cells unchanged
# ============================================================
import xml.etree.ElementTree as ET

ARXIV_API = "http://export.arxiv.org/api/query"
ARXIV_CATS = "(cat:cs.LG OR cat:cs.CL OR cat:cs.CV OR cat:cs.AI OR cat:cs.IR)"
ARXIV_DATE = "submittedDate:[201701010000 TO 202601010000]"
ARXIV_HEADERS = {"User-Agent": "AcademicRAGDemo/1.0 (academic portfolio project)"}

ARXIV_QUERIES = [
    '"machine learning"', '"deep learning"', '"large language models"',
    '"transformer"', '"retrieval augmented generation"',
    '"computer vision"', '"natural language processing"',
]

def arxiv_search(phrase, max_results=25):
    params = {
        "search_query": f"all:{phrase} AND {ARXIV_CATS} AND {ARXIV_DATE}",
        "start": 0, "max_results": max_results, "sortBy": "relevance",
    }
    r = requests.get(ARXIV_API, params=params, headers=ARXIV_HEADERS, timeout=60)
    if r.status_code != 200:
        print("❌ arXiv HTTP", r.status_code)
        return []
    ns = {"a": "http://www.w3.org/2005/Atom"}
    root = ET.fromstring(r.text)
    out = []
    for e in root.findall("a:entry", ns):
        def _txt(tag):
            t = e.find(f"a:{tag}", ns)
            return " ".join((t.text or "").split()) if t is not None else ""
        raw_id = e.find("a:id", ns).text            # .../abs/2005.14165v4
        pid = re.sub(r"v\d+$", "", raw_id.split("/abs/")[-1])  # stable ID
        published = e.find("a:published", ns).text
        out.append({
            "paperId": pid,
            "corpusId": None,
            "title": _txt("title"),
            "abstract": _txt("summary"),
            "year": int(published[:4]) if published else None,
            "authors": [{"name": n.text} for n in e.findall("a:author/a:name", ns) if n.text],
            "url": f"https://arxiv.org/abs/{pid}",
            "openAccessPdf": {"url": f"https://arxiv.org/pdf/{pid}"},
            "citationCount": 0,
            "venue": "arXiv",
            "fieldsOfStudy": ["Computer Science"],
            "search_query": phrase,
        })
    return out

papers, seen = [], set()
for q in ARXIV_QUERIES:
    for p in arxiv_search(q, MAX_PAPERS_PER_QUERY + 5):
        pid = p["paperId"]
        if pid in seen or len(p["abstract"]) < MIN_ABSTRACT_CHARS:
            continue
        papers.append(p); seen.add(pid)
        if len(papers) >= N_PAPERS_TARGET:
            break
    if len(papers) >= N_PAPERS_TARGET:
        break
    print(f"   after '{q}': {len(papers)} papers")
    time.sleep(3)  # arXiv politeness policy

print(f"✅ Collected {len(papers)} papers from arXiv")
papers_df = pd.DataFrame([{
    "paperId": p["paperId"], "title": p["title"],
    "year": p["year"], "pdf_url": p["openAccessPdf"]["url"],
} for p in papers])
display(papers_df.head(10))

   after '"machine learning"': 25 papers
   after '"deep learning"': 50 papers
   after '"large language models"': 75 papers
✅ Collected 80 papers from arXiv


,paperId,title,year,pdf_url
0,2306.04338,Changing Data Sources in the Age of Machine Le...,2023,https://arxiv.org/pdf/2306.04338
1,2304.02381,Physics-Inspired Interpretability Of Machine L...,2023,https://arxiv.org/pdf/2304.02381
2,2201.12150,Learning Curves for Decision Making in Supervi...,2022,https://arxiv.org/pdf/2201.12150
3,2303.15563,Privacy-preserving machine learning for health...,2023,https://arxiv.org/pdf/2303.15563
4,1905.04749,A Benchmark Study of Machine Learning Models f...,2019,https://arxiv.org/pdf/1905.04749
5,1906.01101,MEMe: An Accurate Maximum Entropy Method for E...,2019,https://arxiv.org/pdf/1906.01101
6,2404.12511,Generalizing Machine Learning Evaluation throu...,2024,https://arxiv.org/pdf/2404.12511
7,2402.01393,ALERT-Transformer: Bridging Asynchronous and S...,2024,https://arxiv.org/pdf/2402.01393
8,2111.07508,Public Policymaking for International Agricult...,2021,https://arxiv.org/pdf/2111.07508
9,1706.01513,Beyond Volume: The Impact of Complex Healthcar...,2017,https://arxiv.org/pdf/1706.01513


In [5]:
# ============================================================
# 4. DOWNLOAD ARXIV PDFs — ROBUST VERSION
# ============================================================

import random
from urllib.parse import urlparse

ARXIV_PDF_HOSTS = [
    "https://arxiv.org/pdf/{pid}",
    "https://export.arxiv.org/pdf/{pid}",
]

ARXIV_HEADERS = {
    "User-Agent": (
        "AcademicResearchRAG/1.0 "
        "(educational research project; contact: your_email@example.com)"
    ),
    "Accept": "application/pdf",
}

def safe_name(text, max_len=120):
    text = re.sub(r"[^a-zA-Z0-9._-]+", "_", text or "paper")
    return text[:max_len].strip("._")

def download_pdf(paper, max_retries=4):
    pid = str(paper["paperId"]).strip()

    # arXiv IDs can contain versions, but our stable IDs do not.
    filename = safe_name(pid) + ".pdf"
    path = PDF_DIR / filename

    # Cache
    if path.exists() and path.stat().st_size > 10_000:
        return str(path)

    urls = [u.format(pid=pid) for u in ARXIV_PDF_HOSTS]

    last_error = None

    for url in urls:
        for attempt in range(1, max_retries + 1):

            try:
                r = requests.get(
                    url,
                    headers=ARXIV_HEADERS,
                    timeout=90,
                    allow_redirects=True,
                )

                print(
                    f"[{pid}] "
                    f"HTTP={r.status_code} | "
                    f"final={r.url} | "
                    f"type={r.headers.get('content-type')} | "
                    f"bytes={len(r.content)}"
                )

                # Success
                if (
                    r.status_code == 200
                    and (
                        r.content.startswith(b"%PDF")
                        or "application/pdf" in
                        (r.headers.get("content-type") or "").lower()
                    )
                ):
                    path.write_bytes(r.content)

                    if path.stat().st_size > 10_000:
                        return str(path)

                    path.unlink(missing_ok=True)

                # Retryable responses
                if r.status_code in {429, 500, 502, 503, 504}:
                    wait = min(30, 2 ** (attempt - 1)) + random.random()

                    print(
                        f"⚠️ Retryable HTTP {r.status_code}; "
                        f"sleeping {wait:.1f}s"
                    )

                    time.sleep(wait)
                    continue

                # HTML page / unexpected response
                content_type = (
                    r.headers.get("content-type") or ""
                ).lower()

                if r.status_code == 200 and "pdf" not in content_type:
                    print(
                        f"⚠️ Unexpected content from {r.url}: "
                        f"{r.content[:120]!r}"
                    )

                break

            except requests.RequestException as e:
                last_error = e

                wait = min(30, 2 ** (attempt - 1)) + random.random()

                print(
                    f"⚠️ Network error "
                    f"({attempt}/{max_retries}) for {pid}: {e}"
                )

                if attempt < max_retries:
                    time.sleep(wait)

    print(f"❌ Could not download {pid}: {last_error}")
    return None

In [6]:
# ============================================================
# DOWNLOAD ALL PDFs
# ============================================================

downloaded_papers = []

for p in tqdm(papers, desc="Downloading PDFs"):
    pdf_path = download_pdf(p)

    if pdf_path:
        p["pdf_path"] = pdf_path
        downloaded_papers.append(p)

    # Be polite to arXiv
    time.sleep(1.0)

papers = downloaded_papers

print("=" * 80)
print(f"✅ Downloaded {len(papers)} / {len(papers)} usable PDFs")
print("=" * 80)

if not papers:
    raise RuntimeError(
        "No PDFs were downloaded. "
        "Inspect the diagnostic lines above before continuing."
    )

[2306.04338] HTTP=200 | final=https://arxiv.org/pdf/2306.04338 | type=application/pdf | bytes=152328
[2304.02381] HTTP=200 | final=https://arxiv.org/pdf/2304.02381 | type=application/pdf | bytes=271594
[2201.12150] HTTP=200 | final=https://arxiv.org/pdf/2201.12150 | type=application/pdf | bytes=2496059
[2303.15563] HTTP=200 | final=https://arxiv.org/pdf/2303.15563 | type=application/pdf | bytes=214373
[1905.04749] HTTP=200 | final=https://arxiv.org/pdf/1905.04749 | type=application/pdf | bytes=426578
[1906.01101] HTTP=200 | final=https://arxiv.org/pdf/1906.01101 | type=application/pdf | bytes=441639
[2404.12511] HTTP=200 | final=https://arxiv.org/pdf/2404.12511 | type=application/pdf | bytes=583040
[2402.01393] HTTP=200 | final=https://arxiv.org/pdf/2402.01393 | type=application/pdf | bytes=4279044
[2111.07508] HTTP=200 | final=https://arxiv.org/pdf/2111.07508 | type=application/pdf | bytes=2104966
[1706.01513] HTTP=200 | final=https://arxiv.org/pdf/1706.01513 | type=application/pdf | 

In [8]:
# ============================================================
# 5. PARSE PDFs → SECTION-AWARE + PAGE-AWARE  (defs + loop TOGETHER)
# ============================================================
KNOWN_SECTIONS = {
    "abstract": "Abstract", "introduction": "Introduction",
    "background": "Background", "related work": "Related Work",
    "preliminaries": "Preliminaries", "method": "Methods",
    "methods": "Methods", "methodology": "Methodology",
    "approach": "Approach", "proposed method": "Proposed Method",
    "proposed approach": "Proposed Approach", "model": "Model",
    "experiments": "Experiments", "experimental setup": "Experimental Setup",
    "experimental results": "Experimental Results",
    "evaluation": "Evaluation", "results": "Results",
    "main results": "Results", "results and discussion": "Results and Discussion",
    "analysis": "Analysis", "ablation": "Ablation",
    "ablation study": "Ablation Study", "discussion": "Discussion",
    "conclusion": "Conclusion", "conclusions": "Conclusion",
    "conclusion and future work": "Conclusion",
    "limitations": "Limitations", "future work": "Future Work",
    "ethics": "Ethics", "broader impact": "Broader Impact",
    "acknowledgments": "Acknowledgments", "acknowledgements": "Acknowledgments",
    "references": "References", "appendix": "Appendix",
}
MAX_HEADING_CHARS = 70

def detect_section(line):
    raw = re.sub(r"\s+", " ", line.strip())
    if not raw or raw[-1] in ".,;:" or len(raw) > MAX_HEADING_CHARS:
        return None
    normalized = re.sub(r"[^a-zA-Z0-9 ]", "", raw.lower()).strip()
    if not normalized:
        return None
    if normalized in KNOWN_SECTIONS:
        return KNOWN_SECTIONS[normalized]
    m = re.match(r"^(\d+(?:\.\d+)*)\s+(.+)$", normalized)
    if m and m.group(2).strip() in KNOWN_SECTIONS:
        return KNOWN_SECTIONS[m.group(2).strip()]
    return None

def clean_text(text):
    text = text.replace("\x00", " ")
    text = re.sub(r"-\n(?=\w)", "", text)
    text = re.sub(r"\n+", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    return text.strip()

def parse_pdf(paper):
    doc = fitz.open(paper["pdf_path"])
    pages, current_section = [], "Unknown"
    for page_idx, page in enumerate(doc):
        lines = [l.strip() for l in page.get_text("text").splitlines() if l.strip()]
        segments, sec, buf = [], current_section, []
        for line in lines:
            detected = detect_section(line)
            if detected:
                if buf:
                    segments.append((sec, buf))
                sec, buf = detected, []
            else:
                buf.append(line)
        if buf:
            segments.append((sec, buf))
        current_section = sec
        for sec_name, seg_lines in segments:
            text = clean_text("\n".join(seg_lines))
            if len(text) >= 80:
                pages.append({"page": page_idx + 1, "section": sec_name, "text": text})
    doc.close()
    paper["pages"] = pages
    return paper

valid, failed = [], []
for p in tqdm(papers, desc="Parsing PDFs"):
    try:
        p = parse_pdf(p)
        total_chars = sum(len(x["text"]) for x in p["pages"])
        if total_chars >= 1500:
            valid.append(p)
        else:
            failed.append({"paperId": p["paperId"], "title": p["title"],
                           "reason": "Too little text", "chars": total_chars})
    except Exception as e:
        failed.append({"paperId": p["paperId"], "title": p.get("title"), "reason": str(e)})
papers = valid
print(f"✅ Parsed: {len(papers)} | Rejected: {len(failed)}")
if failed:
    display(pd.DataFrame(failed).head(20))

Parsing PDFs:   0%|          | 0/79 [00:00<?, ?it/s]

✅ Parsed: 79 | Rejected: 0


In [9]:
# ============================================================
# 5.1 PARSING SANITY CHECK
# ============================================================

print("Papers:", len(papers))

if papers:

    sample = papers[0]

    print("\nTITLE:")
    print(sample["title"])

    print("\nPAGES:")
    print(len(sample["pages"]))

    print("\nFIRST PAGE TEXT:")
    print(sample["pages"][5]["text"][:])

else:
    raise RuntimeError("❌ No papers survived PDF parsing.")

Papers: 79

TITLE:
Changing Data Sources in the Age of Machine Learning for Official Statistics

PAGES:
12

FIRST PAGE TEXT:
With the right tools, workforce, technological advances, mindset, and legislative support, these challenges can and
should be manageable. The most challenging piece of the puzzle, however – and one that is more than often ignored –
is the lack of control you can exert over the data sources that are externally gathered. As a national statistics agency,
traditionally, survey data and administrative records that power official statistics are completely under your own control.
But once you start exploiting external data sources to power novel, innovative, complimentary or ersatz statistics, this
lack of control of your data should never be ignored, and if possible, should be front and center on your agenda early on
in the process.
As the popular saying goes: “With great power comes great responsibility” (from Spider-Man, 2002). Having control
and power over your data

In [10]:
# ============================================================
# 6. CHUNKING (per-page + section)
# ============================================================
def word_chunks(text, chunk_size=CHUNK_SIZE_WORDS, overlap=CHUNK_OVERLAP_WORDS):
    words = text.split()
    out, start = [], 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end]).strip()
        if len(chunk.split()) >= MIN_CHUNK_WORDS:
            out.append(chunk)
        if end == len(words):
            break
        start = max(end - overlap, start + 1)
    return out

chunks = []
for paper in papers:
    pid = paper["paperId"]
    authors = [a.get("name", "") for a in (paper.get("authors") or []) if a.get("name")]
    for seg_i, page_info in enumerate(paper["pages"]):   # seg_i = unique per segment
        page_num = page_info["page"]
        section = page_info.get("section", "Unknown")
        for li, txt in enumerate(word_chunks(page_info["text"])):
            chunks.append({
                "chunk_id": f"{pid}_p{page_num}_s{seg_i}_c{li}",   # ← التعديل الوحيد
                "paper_id": pid, "corpus_id": paper.get("corpusId"),
                "title": paper.get("title") or "Untitled",
                "authors": authors, "year": paper.get("year"),
                "page": page_num, "section": section,
                "venue": paper.get("venue"),
                "citation_count": paper.get("citationCount", 0),
                "source_url": paper.get("url"),
                "pdf_url": (paper.get("openAccessPdf") or {}).get("url"),
                "text": txt,
            })

# --- defensive check: catch this class of bug at build time, not in Chroma ---
_ids = [c["chunk_id"] for c in chunks]
assert len(_ids) == len(set(_ids)), f"{len(_ids)-len(set(_ids))} duplicate chunk ids!"

print(f"✅ Created {len(chunks):,} chunks (all IDs unique)")
with open(PAPERS_JSON, "w", encoding="utf-8") as f:
    json.dump(papers, f, ensure_ascii=False, indent=2)
with open(CHUNKS_JSON, "w", encoding="utf-8") as f:
    json.dump(chunks, f, ensure_ascii=False, indent=2)

✅ Created 4,704 chunks (all IDs unique)


In [11]:
# ============================================================
# 6.5 RESUME AFTER RESTART  (run INSTEAD of cells 3-6)
# ============================================================
if "chunks" not in globals() or not chunks:
    with open(PAPERS_JSON, "r", encoding="utf-8") as f:
        papers = json.load(f)
    with open(CHUNKS_JSON, "r", encoding="utf-8") as f:
        chunks = json.load(f)
    print(f"✅ Loaded {len(papers)} papers / {len(chunks):,} chunks from disk")
chunk_texts = [c["text"] for c in chunks]

In [12]:
# ============================================================
# 7. BM25
# ============================================================
def tokenize(text):
    return re.findall(r"(?u)\b\w+\b", text.lower())

chunk_texts = [c["text"] for c in chunks]
tokenized_corpus = [tokenize(t) for t in chunk_texts]
bm25 = BM25Okapi(tokenized_corpus)
print("✅ BM25 index ready")

✅ BM25 index ready


In [13]:
# ============================================================
# 8. EMBEDDINGS (cached + auto-invalidated) + CHROMADB
# ============================================================
from chromadb import PersistentClient
from chromadb.config import Settings

# --- Auto-invalidate stale embeddings on ANY config change ---
RUN_CFG = {
    "parser": "section-aware-v2",
    "embedding_model": EMBEDDING_MODEL,
    "chunk_size": CHUNK_SIZE_WORDS,
    "chunk_overlap": CHUNK_OVERLAP_WORDS,
    "min_chunk": MIN_CHUNK_WORDS,
}
_prev = json.loads(RUN_CFG_JSON.read_text()) if RUN_CFG_JSON.exists() else None
if _prev != RUN_CFG:
    print("⚠️ Config changed → invalidating embeddings cache (will re-encode)")
    EMBEDDINGS_NPY.unlink(missing_ok=True)
    RUN_CFG_JSON.write_text(json.dumps(RUN_CFG, indent=2))

embedder = SentenceTransformer(EMBEDDING_MODEL)

embeddings = None
if EMBEDDINGS_NPY.exists():
    emb = np.load(EMBEDDINGS_NPY)
    if emb.shape[0] == len(chunks):
        embeddings = emb
        print("✅ Using cached embeddings:", embeddings.shape)
    else:
        print("⚠️ stale embeddings (wrong count) — re-encoding")

if embeddings is None:
    embeddings = embedder.encode(
        chunk_texts, batch_size=64, show_progress_bar=True,
        normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    np.save(EMBEDDINGS_NPY, embeddings)
    print("✅ Encoded + cached:", embeddings.shape)

client = PersistentClient(path=str(CHROMA_DIR),
                          settings=Settings(anonymized_telemetry=False))
try:
    client.delete_collection("academic_papers")
except Exception:
    pass
collection = client.get_or_create_collection(
    name="academic_papers", metadata={"hnsw:space": "cosine"}
)

ids = [c["chunk_id"] for c in chunks]
metadatas = [{
    "paper_id": c["paper_id"],
    "corpus_id": str(c.get("corpus_id") or ""),
    "title": c["title"],
    "page": int(c["page"]),
    "year": int(c["year"]) if c.get("year") else 0,
    "section": c.get("section", "Unknown"),
    "authors": ", ".join(c["authors"][:5]),
    "source_url": c.get("source_url") or "",
    "pdf_url": c.get("pdf_url") or "",
} for c in chunks]

BATCH = 5000
for i in range(0, len(ids), BATCH):
    collection.add(ids=ids[i:i+BATCH], documents=chunk_texts[i:i+BATCH],
                   embeddings=embeddings[i:i+BATCH].tolist(),
                   metadatas=metadatas[i:i+BATCH])

print(f"✅ Chroma ready: {collection.count():,} chunks")

⚠️ Config changed → invalidating embeddings cache (will re-encode)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

Batches:   0%|          | 0/74 [00:00<?, ?it/s]

✅ Encoded + cached: (4704, 1024)
✅ Chroma ready: 4,704 chunks


In [14]:
# ============================================================
# 9. HYBRID SEARCH — RRF(k=60)
# ============================================================
ID_TO_IDX = {c["chunk_id"]: i for i, c in enumerate(chunks)}

def bm25_search(query, top_k=BM25_TOP_K):
    raw = bm25.get_scores(tokenize(query))
    idx = np.argsort(raw)[::-1][:top_k]
    return [{"idx": int(i), "text": chunks[i]["text"], "chunk": chunks[i]}
            for i in idx]

def dense_search(query, top_k=DENSE_TOP_K):
    q_emb = embedder.encode([query], normalize_embeddings=True,
                            convert_to_numpy=True)[0].astype("float32")
    res = collection.query(query_embeddings=[q_emb.tolist()],
                           n_results=min(top_k, len(chunks)),
                           include=["distances"])
    results = []
    for cid, dist in zip(res["ids"][0], res["distances"][0]):
        i = ID_TO_IDX[cid]
        results.append({"idx": int(i), "text": chunks[i]["text"], "chunk": chunks[i]})
    return results

def hybrid_search(query, top_k=HYBRID_TOP_K, rrf_k=60):

    fused, info = {}, {}
    for rank, r in enumerate(bm25_search(query), 1):
        fused[r["idx"]] = fused.get(r["idx"], 0.0) + 1.0 / (rrf_k + rank)
        info[r["idx"]] = r
    for rank, r in enumerate(dense_search(query), 1):
        fused[r["idx"]] = fused.get(r["idx"], 0.0) + 1.0 / (rrf_k + rank)
        info[r["idx"]] = r

    results = []
    for idx, s in fused.items():
        r = dict(info[idx])
        r["hybrid_score"] = s
        results.append(r)
    results.sort(key=lambda x: x["hybrid_score"], reverse=True)
    return results[:top_k]

print("✅ Hybrid retriever (RRF) ready")

✅ Hybrid retriever (RRF) ready


In [15]:
# ============================================================
# 10. CROSS-ENCODER RE-RANKING
# ============================================================
reranker = CrossEncoder(RERANKER_MODEL)

def rerank(query, candidates, top_k=RERANK_TOP_K):
    pairs = [(query, c["text"]) for c in candidates]
    scores = reranker.predict(pairs, show_progress_bar=False)
    ranked = []
    for c, score in zip(candidates, scores):
        item = dict(c)
        item["rerank_score"] = float(score)
        ranked.append(item)
    ranked.sort(key=lambda x: x["rerank_score"], reverse=True)
    return ranked[:top_k]

print("✅ Cross-Encoder reranker ready")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

✅ Cross-Encoder reranker ready


In [16]:
# ============================================================
# 11. GENERATOR — Gemini API + local flan-t5 fallback
# ============================================================
USE_API = True

if USE_API:
    GEMINI_API_KEY = ""
    try:  # Colab
        from google.colab import userdata
        GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        pass
    if not GEMINI_API_KEY:
        try:  # Kaggle
            from kaggle_secrets import UserSecretsClient
            GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY")
        except Exception:
            pass
    if not GEMINI_API_KEY:
        GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")  # local
    if not GEMINI_API_KEY:
        raise RuntimeError(
            "GEMINI_API_KEY not found!\n"
            "Kaggle: Add-ons → Secrets → Add GEMINI_API_KEY → Attach to notebook\n"
            "Colab:  🔑 icon → Add secret → GEMINI_API_KEY\n"
            "Local:  os.environ['GEMINI_API_KEY'] = 'AIza...'"
        )
    from google import genai
    from google.genai import types
    llm_client = genai.Client(api_key=GEMINI_API_KEY)
    tokenizer, generator, device = None, None, "cpu"
    print("✅ Generator:", GEMINI_MODEL, "via Gemini API")
else:
    from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
    generator = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL).to(device)
    generator.eval()
    print("✅ Generator: local", GENERATOR_MODEL, "on", device)

✅ Generator: gemini-3.5-flash-lite via Gemini API


In [17]:
# ============================================================
# 12. ANSWER + CITATIONS (validator + diagnostics)
# ============================================================
CTX_TOP_K = 6        # evidence blocks the generator sees
CTX_MAX_WORDS = 150  # per block
CIT_RE = re.compile(r"\[S(\d+)\]")

def trim_to_words(text, max_words):
    words = text.split()
    return " ".join(words[:max_words]) + (" …" if len(words) > max_words else "")

def make_context(ranked):
    blocks = []
    for j, r in enumerate(ranked[:CTX_TOP_K], 1):
        c = r["chunk"]
        blocks.append(
            f"[S{j}] Title: {c['title']} ({c.get('year') or 'n.d.'}, p.{c['page']})\n"
            f"{trim_to_words(c['text'], CTX_MAX_WORDS)}"
        )
    return "\n\n".join(blocks)

def build_prompt(query, context):
    return f""" You are a rigorous academic research assistant.

Answer the question using ONLY the numbered evidence snippets from the research corpus.

Rules:
1. Never use outside knowledge. Never invent facts, numbers, methods, or citations.
2. After every factual claim, add source tags like [S1] or [S2].
3. Synthesize across sources; do not dump the sources one by one.
4. If the evidence is insufficient, reply exactly: "The corpus does not provide enough evidence."

Question: {query}

Evidence:
{context}

Answer:
"""

def _llm_chat(prompt, max_retries=3):
    backoff = 5
    for attempt in range(1, max_retries + 1):
        try:
            resp = llm_client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=(
                        "You are a precise academic RAG assistant. "
                        "Follow the [S#] citation format exactly."
                    ),
                    temperature=0.0,
                    max_output_tokens=600,
                ),
            )
            try:
                text = resp.text
            except Exception:
                text = None
            if not text or not text.strip():
                print("⚠️ Gemini returned empty/blocked response")
                return ""
            return text.strip()
        except Exception as e:
            print(f"⚠️ Gemini error (try {attempt}/{max_retries}): {e}")
            if attempt == max_retries:
                raise
            time.sleep(backoff); backoff *= 2

def generate_answer(query, ranked):
    if not ranked:
        return "I could not find supporting evidence in the corpus.", ""
    prompt = build_prompt(query, make_context(ranked))
    if USE_API:
        answer = _llm_chat(prompt)
    else:
        inputs = tokenizer(prompt, return_tensors="pt",
                           truncation=True, max_length=512).to(device)
        with torch.no_grad():
            out = generator.generate(**inputs, max_new_tokens=180, do_sample=False,
                                     num_beams=4, repetition_penalty=1.08)
        answer = tokenizer.decode(out[0], skip_special_tokens=True).strip()

    # --- Citation validator: catches format-level hallucination ---
    cited = sorted({int(m) for m in CIT_RE.findall(answer)})
    status = "✅"
    if not cited:
        status = "⚠️ NO [S#] TAGS — treat answer as ungrounded"
    elif any(c > CTX_TOP_K for c in cited):
        status = f"⚠️ INVALID TAGS {[c for c in cited if c > CTX_TOP_K]}"
    return answer, f"{status} | cited={cited}"

def citation_table(ranked):
    """ S# here maps EXACTLY to what the generator saw — built after re-ranking."""
    rows = []
    for j, r in enumerate(ranked[:CTX_TOP_K], 1):
        c = r["chunk"]
        at = ", ".join(c["authors"][:3])
        if len(c["authors"]) > 3:
            at += " et al."
        rows.append({"source": f"S{j}", "title": c["title"], "authors": at,
                     "year": c.get("year"), "page": c["page"],
                     "rerank_score": round(r["rerank_score"], 4),
                     "url": c.get("source_url") or c.get("pdf_url") or ""})
    return pd.DataFrame(rows)

In [18]:
# ============================================================
# 13. END-TO-END ASK (degraded mode if LLM fails)
# ============================================================
def ask(query, verbose=True):
    candidates = hybrid_search(query, HYBRID_TOP_K)
    ranked = rerank(query, candidates, RERANK_TOP_K)
    sources = citation_table(ranked)
    try:
        answer, diag = generate_answer(query, ranked)
    except Exception as e:
        answer = ("⚠️ LLM unavailable right now — but retrieval + re-ranking "
                  "are fully functional. Top evidence below.")
        diag = f"❌ generation failed: {e}"

    if verbose:
        print("\n" + "="*90)
        print("QUESTION:", query)
        print("="*90)
        print("\nANSWER")
        print(answer)
        if diag:
            print("\n" + diag)
        print("\n" + "-"*90)
        print("SOURCES (exactly what the generator saw)")
        display(sources[["source", "title", "year", "page", "rerank_score"]])

    return {"answer": answer, "sources": sources, "retrieved": ranked}

print("✅ Full RAG pipeline ready")

✅ Full RAG pipeline ready


In [19]:
# ============================================================
# 14. TEST
# ============================================================
q = "What are the main limitations of transformer-based models discussed in these papers?"
result = ask(q)


QUESTION: What are the main limitations of transformer-based models discussed in these papers?

ANSWER
Based on the provided papers, the limitations and challenges associated with transformer-based and large language models include:

* **Resource and Cost Constraints:** Large language models require substantial computational resources and energy, resulting in high operational costs for training and deployment [S4]. Additionally, resource limits can prevent scaling to larger models (such as 65B and 70B LLama models) [S2].
* **General Model Issues:** LLMs suffer from inherent limitations such as hallucinations, inconsistent outputs, and context limitations [S4].
* **Hyper-parameter Dependency:** Boundary constraint alignment losses rely on a hyper-parameter $\beta$ that significantly impacts performance, requiring a validation set and search overhead to find the optimal value [S2]. 
* **Training and Architectural Challenges:** Training pure transformer-based backbones, especially comple

,source,title,year,page,rerank_score
0,S1,Transformer in Transformer as Backbone for Dee...,2022,8,3.6214
1,S2,Making Large Language Models Better Reasoners ...,2023,10,1.4800
2,S3,ALERT-Transformer: Bridging Asynchronous and S...,2024,9,1.1310
3,S4,A Survey of AIOps in the Era of Large Language...,2025,24,0.2674
4,S5,PePR: Performance Per Resource Unit as a Metri...,2024,4,-0.0122
5,S6,Transformer in Transformer as Backbone for Dee...,2022,19,-0.1133


In [ ]:
# ============================================================
# 15. OPTIONAL: INTERACTIVE CHAT
# ============================================================
print("Academic Research Assistant — type 'exit' to stop.\n")
while True:
    q = input("You: ").strip()
    if q.lower() in {"exit", "quit"}:
        print("Bye.")
        break
    if not q:
        continue
    ask(q)

In [21]:
res = ask("What do these papers say about quantum machine learning?", verbose=False)
for tag, r in zip(["S1","S5"], res["retrieved"][:6]):
    c = r["chunk"]
    print("="*90, f"\n{tag} | p.{c['page']}")
    idx = c["text"].lower().find("quantum")
    print("..." if idx < 0 else c["text"][max(0,idx-200):idx+400])

S1 | p.48
umPyro. arXiv preprint arXiv:1912.11554 2019. 229. Broughton, M.; Verdon, G.; McCourt, T.; Martinez, A.J.; Yoo, J.H.; Isakov, S.V.; Massey, P.; Niu, M.Y.; Halavati, R.; Peters, E.; others. TensorFlow Quantum: A Software Framework for Quantum Machine Learning. arXiv preprint arXiv:2003.02989 2020. 230. Silver, D.; Hubert, T.; Schrittwieser, J.; Antonoglou, I.; Lai, M.; Guez, A.; Lanctot, M.; Sifre, L.; Kumaran, D.; Graepel, T.; others. Mastering chess and Shogi by self-play with a general reinforcement learning algorithm. arXiv preprint arXiv:1712.01815 2017. 231. Vinyals, O.; Babuschkin, I.; C
S5 | p.36
stonishing breakthroughs enabled by RL in the upcoming years. Also, we hope 57 https://github.com/huggingface/transformers 58 https://github.com/rapidsai/cuml 59 https://github.com/stan-dev/pystan 60 Quantum computing is an approach to computing based on quantum mechanics. In a classical computer, the basic unit of information is the bit, which is a binary variable that can as

In [22]:
# ============================================================
# 16. MANIFEST
# ============================================================
manifest = {
    "num_papers": len(papers),
    "num_chunks": len(chunks),
    "embedding_model": EMBEDDING_MODEL,
    "reranker_model": RERANKER_MODEL,
    "generator_model": GEMINI_MODEL if USE_API else GENERATOR_MODEL,
    "fusion": "RRF(k=60)",
    "chunk_size_words": CHUNK_SIZE_WORDS,
    "chunk_overlap_words": CHUNK_OVERLAP_WORDS,
    "min_chunk_words": MIN_CHUNK_WORDS,
    "bm25_top_k": BM25_TOP_K, "dense_top_k": DENSE_TOP_K,
    "hybrid_top_k": HYBRID_TOP_K, "rerank_top_k": RERANK_TOP_K,
    "ctx_top_k": CTX_TOP_K, "ctx_max_words": CTX_MAX_WORDS,
}
with open(ARTIFACT_DIR / "manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)
print(json.dumps(manifest, indent=2))
print("✅ Saved under:", BASE_DIR)

{
  "num_papers": 79,
  "num_chunks": 4704,
  "embedding_model": "BAAI/bge-m3",
  "reranker_model": "cross-encoder/ms-marco-MiniLM-L-6-v2",
  "generator_model": "gemini-3.5-flash-lite",
  "fusion": "RRF(k=60)",
  "chunk_size_words": 220,
  "chunk_overlap_words": 40,
  "min_chunk_words": 40,
  "bm25_top_k": 30,
  "dense_top_k": 30,
  "hybrid_top_k": 40,
  "rerank_top_k": 10,
  "ctx_top_k": 6,
  "ctx_max_words": 150
}
✅ Saved under: /content/academic_rag
